# fastfuncstuff — Colab Benchmark

Runs the FFS benchmark pipeline against pre-computed reference outputs.

**Setup:**
1. Copy your `ds005165-download` folder (with existing AFNI/ref outputs) to Google Drive
2. Copy your `fastfuncstuff` repo to Google Drive (or `git clone` it below)
3. Update the paths in the config cell
4. Runtime → Change runtime type → **T4 GPU** (or better)
5. Run All

Only FFS tools are re-run (`-force-ffs`). Reference tool outputs are read from the
copied data — no AFNI/MATLAB/FSL required.

After the run, the updated `benchmark_cache.json` is copied back to Drive.
Import it locally with:
```
ffs_benchmark -import-cache /path/to/benchmark_cache.json
```

In [ ]:
# ============================================================
# CONFIGURE THESE
# ============================================================

# Path to your fastfuncstuff repo on Google Drive
FFS_REPO = "/content/drive/MyDrive/code/fastfuncstuff"

# Path to ds005165-download on Google Drive (with ref outputs already computed)
DRIVE_DATA = "/content/drive/MyDrive/data/ds005165-download"

# Local copy on Colab disk (much faster I/O than Drive during the run)
LOCAL_DATA = "/content/ds005165-download"

# Stages to run — None means all FFS stages.
# To run a subset: STAGES = "glm,ica,glmsingle_hrf"
STAGES = None

## 1. Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# System deps: pigz (fast gzip), zstd (nii.zst support)
!apt-get -qq install pigz zstd

In [ ]:
# Install fastfuncstuff from your Google Drive copy
!pip install -q -e "{FFS_REPO}"

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found — FFS tools will run on CPU (slower)")

import fastfuncstuff
print("fastfuncstuff imported OK")

## 2. Copy benchmark data to local disk

Google Drive I/O is slow — copy the whole dataset to `/content/` first.
This includes all pre-computed reference outputs so the benchmark can validate immediately.

In [ ]:
import shutil
import time
from pathlib import Path

local = Path(LOCAL_DATA)

if local.exists() and (local / "sub-01").exists():
    print(f"Already present: {LOCAL_DATA}")
else:
    print(f"Copying {DRIVE_DATA} → {LOCAL_DATA} ...")
    t0 = time.time()
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    elapsed = time.time() - t0
    size_gb = sum(f.stat().st_size for f in local.rglob("*") if f.is_file()) / 1e9
    print(f"Done in {elapsed:.0f}s  ({size_gb:.2f} GB)")

# Sanity check
cache = local / "benchmark_cache.json"
print(f"Cache present: {cache.exists()}  ({cache.stat().st_size / 1e3:.1f} KB)" if cache.exists() else "No cache yet — will be created after the run")

## 3. Run FFS benchmark

Only FFS stages are (re-)run. Reference outputs are read from the copied data directory.
Timings and hardware info are appended to `benchmark_cache.json`.

In [ ]:
stages_flag = f"-stages {STAGES}" if STAGES else ""

!ffs_benchmark \
    -data-dir "{LOCAL_DATA}" \
    -force-ffs \
    -report \
    {stages_flag}

## 4. Inspect cache

In [ ]:
!ffs_benchmark -data-dir "{LOCAL_DATA}" -list-cache

## 5. Copy cache back to Drive

Saves the updated `benchmark_cache.json` back to Drive so you can import the
Colab timings into your local cache with:
```
ffs_benchmark -import-cache /path/to/benchmark_cache_colab.json
```

In [ ]:
import shutil
from pathlib import Path

local_cache = Path(LOCAL_DATA) / "benchmark_cache.json"
drive_cache = Path(DRIVE_DATA) / "benchmark_cache_colab.json"

if local_cache.exists():
    drive_cache.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_cache, drive_cache)
    print(f"Saved: {drive_cache}")
    print()
    print("Import locally with:")
    print(f"  ffs_benchmark -import-cache '{drive_cache}'")
else:
    print("No cache found — did the benchmark run complete?")